# Remote diffusion v8.5 - Google Drive results
Colab publishes binary latent results larger than 500 KiB to MyDrive/ComfyUI/remote_results. Smaller results use HTTP.
The local plugin downloads them through an authorized rclone remote.
Cloudflare carries job submission, status and progress. Uploaded inputs still use the tunnel.
No rclone credentials or connection tokens are included in the archive.

**v9 (Anima 改版)**:在 v8.5 基础上增加 Anima 模型下载、在 Colab 上训练 LoRA、训练时不被空闲释放误杀。
远程采样部分(第 1–11 格)没有改动协议,仍需使用 v8.5 插件 ZIP。


In [ ]:
import sys, subprocess, json, os, time, re, zipfile
from pathlib import Path, PurePosixPath

subprocess.run(['nvidia-smi'], check=True)
COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(COMFY / 'requirements.txt')], check=True)
print('ComfyUI dependencies ready.')

## 上传连接插件
选择下载到本机的 `ComfyUI-RemoteDiffusion-v8.5.zip`。不要上传 Discord 配置或其他密钥文件。

In [ ]:
from google.colab import files
uploaded = files.upload()
package_name = 'ComfyUI-RemoteDiffusion-v8.5.zip'
if package_name not in uploaded:
    raise ValueError('Please upload ' + package_name)
with zipfile.ZipFile(package_name) as archive:
    for member in archive.namelist():
        parts = PurePosixPath(member).parts
        if not parts or parts[0] not in ('ComfyUI-RemoteDiffusion', 'ComfyUI-SeedVR2-1.4B') or '..' in parts or PurePosixPath(member).is_absolute():
            raise ValueError('Unexpected archive path')
    archive.extractall(COMFY / 'custom_nodes')
PLUGIN = COMFY / 'custom_nodes' / 'ComfyUI-RemoteDiffusion'
import secrets
connection_config = {'url': '', 'token': secrets.token_urlsafe(32), 'proxy': 'http://127.0.0.1:7890', 'drive_remote': 'comfy_drive:ComfyUI/remote_results'}
print('Plugin installed. New session key generated without displaying it.')

## 挂载 Drive 并指定模型目录
将原来的 checkpoint 和使用的 LoRA 上传至自己的 Google Drive：

- `MyDrive/ComfyUI/models/checkpoints/xaviernocturne_v3.safetensors`
- `MyDrive/ComfyUI/models/loras/你的LoRA.safetensors`

这里不需要上传 VAE、CLIP、提示词文件或原图。模型较大，首次上传需要时间。
如果模型放在其他目录，修改下方 `MODELS_DIR`。文件名与本机节点配置必须一致。
SeedVR2 远程采样还需要上传扩散权重到 `MyDrive/ComfyUI/models/diffusion_models/seedvr2_distill_6L_1.4B_sharp_fp16_comfyui.safetensors`。VAE 留在本机，不需要上传。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODELS_DIR = '/content/drive/MyDrive/ComfyUI/models' #@param {type:'string'}
CHECKPOINT_NAME = 'xaviernocturne_v3.safetensors' #@param {type:'string'}  # 只用 Anima 时可以留空
model_root = Path(MODELS_DIR)
if CHECKPOINT_NAME and not (model_root / 'checkpoints' / CHECKPOINT_NAME).is_file():
    raise FileNotFoundError('Upload the checkpoint first: ' + str(model_root / 'checkpoints' / CHECKPOINT_NAME))
model_paths_file = COMFY / 'extra_models_remote_diffusion.yaml'
model_paths_file.write_text(json.dumps({'remote_models': {'base_path': str(model_root), 'checkpoints': 'checkpoints', 'loras': 'loras', 'diffusion_models': 'diffusion_models'}}))
print('Model directories configured.')
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/ComfyUI/remote_results'
Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print('Drive result directory ready.')


## 启动受密钥保护的计算后端
仅加载连接插件，并禁用合作方 API 节点。此后端只用于远程采样，不提供免登录网页。
不要同时从本机提交相同任务多次。本机中断等待不保证取消 Colab 上已排队的任务。

In [ ]:
import requests
if 'worker' in globals() and worker.poll() is None:
    raise RuntimeError('Worker is already running. Do not start it twice.')
worker_env = os.environ.copy()
worker_env['REMOTE_DIFFUSION_SERVER'] = '1'
worker_env['REMOTE_DIFFUSION_DRIVE_DIR'] = DRIVE_RESULTS_DIR
worker_env['REMOTE_DIFFUSION_TOKEN'] = connection_config['token']
worker_log = open('/content/remote_diffusion_worker.log', 'w')
worker = subprocess.Popen([
    sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
    '--offline', '--preview-method', 'none',
    '--disable-all-custom-nodes', '--whitelist-custom-nodes', 'ComfyUI-RemoteDiffusion', 'ComfyUI-SeedVR2-1.4B',
    '--extra-model-paths-config', str(model_paths_file),
], cwd=COMFY, env=worker_env, stdout=worker_log, stderr=subprocess.STDOUT)
worker_headers = {'Authorization': 'Bearer ' + connection_config['token']}
for attempt in range(90):
    if worker.poll() is not None:
        print(Path('/content/remote_diffusion_worker.log').read_text()[-6000:])
        raise RuntimeError('Worker failed to start.')
    try:
        health = requests.get('http://127.0.0.1:8188/system_stats', headers=worker_headers, timeout=3)
        if health.status_code == 200:
            print('Authenticated ComfyUI worker ready.')
            break
    except requests.ConnectionError:
        pass
    time.sleep(2)
else:
    raise TimeoutError('Worker not ready; inspect /content/remote_diffusion_worker.log')
queue_health = requests.get("http://127.0.0.1:8188/remote_pipeline/status", headers=worker_headers, timeout=5)
queue_health.raise_for_status()
assert queue_health.json().get("protocol_version") == 3, "Install the v8.5 ZIP and restart the cloud worker."
print("Continuous cloud scheduler ready:", queue_health.json()["tasks_remaining"])


## 建立 HTTPS 连接并下载本机配置
使用 Cloudflare 临时隧道连接本机与 Colab。后端的全部 HTTP 请求都需要连接密钥。
下面会下载 `config.json`，复制到本机插件目录，覆盖该插件原来的配置文件。
隧道地址可能随会话变化，下次启动需重新下载配置。

In [ ]:
def wait_remote_endpoint(url, headers, tunnel_process, timeout_seconds=180):
    deadline = time.monotonic() + timeout_seconds
    last_error = 'No response received'
    while time.monotonic() < deadline:
        if 'worker' in globals() and worker.poll() is not None:
            raise RuntimeError('ComfyUI worker exited while waiting for the tunnel. Inspect /content/remote_diffusion_worker.log and restart the worker.')
        if tunnel_process.poll() is not None:
            raise RuntimeError('Tunnel exited. Inspect /content/remote_diffusion_tunnel.log and rerun this cell.')
        try:
            response = requests.get(url + '/object_info/RemoteDiffusionInputs', headers=headers, timeout=(5, 10))
        except (requests.ConnectionError, requests.Timeout) as error:
            last_error = type(error).__name__ + ': DNS or tunnel connection unavailable'
        else:
            if response.status_code in (502, 503, 504, 530):
                last_error = 'HTTP ' + str(response.status_code)
            else:
                response.raise_for_status()
                try:
                    info = response.json()
                except requests.exceptions.JSONDecodeError as error:
                    raise RuntimeError('Tunnel returned non-JSON content. Verify the endpoint and tunnel log.') from error
                if 'RemoteDiffusionInputs' not in info:
                    raise RuntimeError('RemoteDiffusionInputs is missing. Check the uploaded plugin and restart the worker.')
                return
        time.sleep(3)
    raise TimeoutError('Tunnel did not become ready within 180 seconds (' + last_error + '). Inspect /content/remote_diffusion_tunnel.log. If DNS still fails, restart only the tunnel with tunnel.terminate(); tunnel.wait(), then rerun this cell.')

# Check the local origin before creating or checking the public tunnel.
if 'worker' in globals() and worker.poll() is not None:
    raise RuntimeError('ComfyUI worker has exited. Inspect /content/remote_diffusion_worker.log and rerun the worker startup cell before this cell.')
try:
    origin = requests.get('http://127.0.0.1:8188/object_info/RemoteDiffusionInputs', headers=worker_headers, timeout=5)
except requests.ConnectionError as error:
    raise RuntimeError('ComfyUI is not listening on Colab port 8188. Inspect /content/remote_diffusion_worker.log and run the worker startup cell first. Restarting the tunnel will not fix this.') from error
origin.raise_for_status()
if 'RemoteDiffusionInputs' not in origin.json():
    raise RuntimeError('Local worker is missing RemoteDiffusionInputs. Check the plugin ZIP and restart the worker.')

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    binary_url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    with requests.get(binary_url, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with cloudflared.open('wb') as output:
            for chunk in response.iter_content(1024 * 1024):
                output.write(chunk)
    cloudflared.chmod(0o755)
if 'tunnel' not in globals() or tunnel.poll() is not None:
    tunnel_log = open('/content/remote_diffusion_tunnel.log', 'w')
    tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
else:
    print('Reusing the running tunnel.')
for attempt in range(60):
    log = Path('/content/remote_diffusion_tunnel.log').read_text()
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if match:
        connection_config['url'] = match.group(0)
        break
    if tunnel.poll() is not None:
        print(log[-3000:])
        raise RuntimeError('Tunnel failed.')
    time.sleep(1)
else:
    raise TimeoutError('Tunnel URL not ready; inspect the tunnel log.')
print('Remote endpoint:', connection_config['url'])
wait_remote_endpoint(connection_config['url'], worker_headers, tunnel)
print('Authenticated remote endpoint ready.')
config_download = Path('/content/config.json')
config_download.write_text(json.dumps(connection_config, indent=2))
files.download(str(config_download))
print('Copy config.json into the LOCAL ComfyUI-RemoteDiffusion folder. Do not share this file.')

## 空闲 15 分钟自动释放 GPU
运行下一格后，后台监视器会检查远程任务队列和最新完成任务。连续空闲 15 分钟时调用 `runtime.unassign()` 结束整个 Colab 运行时，正在运行或排队的任务不会触发释放。修改 `IDLE_MINUTES` 后重新运行此格即可调整时长。

本机编码提示词、修脸、放大不算云端活动。状态查询失败时暂停空闲判定；释放失败会显示错误，请手动断开并删除运行时。只在此 Notebook 会话中使用此 GPU 服务。


In [ ]:
import threading
from google.colab import runtime

IDLE_MINUTES = 30  # Change this value to adjust the idle timeout.
POLL_SECONDS = 10

def watch_idle_runtime(stop_event, idle_seconds, get_status, release, clock=time.monotonic, poll_seconds=POLL_SECONDS):
    last_active = clock()
    last_completed = None
    while not stop_event.is_set():
        try:
            status = get_status()
        except requests.RequestException as error:
            # A failed check cannot establish that the worker is idle.
            last_active = clock()
            print('Idle check failed; release postponed:', type(error).__name__)
        else:
            now = clock()
            completed = status['last_completed']
            if status['tasks_remaining'] or completed != last_completed:
                last_active = now
            last_completed = completed
            if now - last_active >= idle_seconds:
                print('Worker idle for', IDLE_MINUTES, 'minutes; releasing Colab runtime.')
                try:
                    release()
                except Exception as error:
                    print('Runtime release failed. Disconnect and delete it manually:', type(error).__name__)
                return
        if stop_event.wait(poll_seconds):
            return

def get_worker_status():
    response = requests.get('http://127.0.0.1:8188/remote_diffusion/status', headers=worker_headers, timeout=5)
    response.raise_for_status()
    status = response.json()
    # v9: a running LoRA training process (started in the training section) counts as activity,
    # otherwise the runtime would be released in the middle of a long training run.
    train_proc = globals().get('TRAIN_PROC')
    if train_proc is not None and train_proc.poll() is None:
        status['tasks_remaining'] = (status.get('tasks_remaining') or 0) + 1
    return status

# Verify the companion plugin supports idle monitoring before starting.
get_worker_status()
if 'idle_release_stop' in globals():
    idle_release_stop.set()
    idle_release_thread.join(timeout=6)
    if idle_release_thread.is_alive():
        raise RuntimeError('Previous idle monitor has not stopped. Retry this cell.')
idle_release_stop = threading.Event()
idle_release_thread = threading.Thread(
    target=watch_idle_runtime,
    args=(idle_release_stop, IDLE_MINUTES * 60, get_worker_status, runtime.unassign),
    daemon=True,
)
idle_release_thread.start()
print('Automatic runtime release enabled: idle for', IDLE_MINUTES, 'minutes. Running and queued jobs reset the timer.')


## (v9) Anima:下载模型到 Drive
模型来自 [circlestone-labs/Anima](https://huggingface.co/circlestone-labs/Anima)(非商业许可)。
文件放进 Drive 后以后每次开新会话都不用重新下载:

- `models/diffusion_models/anima-base-v1.0.safetensors`:远程采样和训练都用
- `models/text_encoders/qwen_3_06b_base.safetensors`、`models/vae/qwen_image_vae.safetensors`:训练用;按原设计,远程采样时文本编码和 VAE 仍留在本机

如果 HF 要求登录,在 Colab 左侧「密钥」添加 `HF_TOKEN`。

In [ ]:
import shutil
from huggingface_hub import HfApi, hf_hub_download
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

ANIMA_REPO = 'circlestone-labs/Anima'
_repo_files = HfApi().list_repo_files(ANIMA_REPO)
def fetch_anima(name, subdir):
    dest = model_root / subdir / name
    if dest.is_file():
        print('exists:', dest)
        return dest
    hits = [f for f in _repo_files if f.endswith(name)]
    assert hits, name + ' not found in ' + ANIMA_REPO
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = hf_hub_download(ANIMA_REPO, hits[0], local_dir='/content/hf_tmp')
    shutil.copyfile(tmp, dest)
    os.remove(tmp)
    print('downloaded:', dest)
    return dest

DIT_PATH  = fetch_anima('anima-base-v1.0.safetensors', 'diffusion_models')
QWEN_PATH = fetch_anima('qwen_3_06b_base.safetensors', 'text_encoders')
VAE_PATH  = fetch_anima('qwen_image_vae.safetensors', 'vae')

## (v9) 安装训练器 sd-scripts
装在独立的 venv 里(继承系统包),避免升级/降级系统里的 torch 等包而弄坏正在运行的 ComfyUI worker。

In [ ]:
SD_DIR = Path('/content/sd-scripts')
if not SD_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kohya-ss/sd-scripts.git', str(SD_DIR)], check=True)
VENV = Path('/content/sd_venv')
if not (VENV / 'bin' / 'python').exists():
    try:
        subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
        subprocess.run([str(VENV / 'bin' / 'python'), '-m', 'pip', '--version'], check=True)
    except subprocess.CalledProcessError:
        subprocess.run(['rm', '-rf', str(VENV)])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)
        subprocess.run([sys.executable, '-m', 'virtualenv', '--system-site-packages', str(VENV)], check=True)
VPY = str(VENV / 'bin' / 'python')
subprocess.run([VPY, '-m', 'pip', 'install', '-q', '-r', str(SD_DIR / 'requirements.txt')], cwd=SD_DIR, check=True)
subprocess.run([VPY, '-m', 'pip', 'install', '-q', 'bitsandbytes'], check=True)
for label, py in (('system', sys.executable), ('venv', VPY)):
    out = subprocess.run([py, '-c', 'import torch;print(torch.__version__)'], capture_output=True, text=True).stdout.strip()
    print(label, 'torch', out)
print('如果两个 torch 版本不一致,说明 requirements 覆盖了 torch;先确认训练能跑再决定是否处理。')

## (v9) 准备训练数据
把数据放到 Drive:`MyDrive/ComfyUI/train_data/<RUN_NAME>/`,里面是图片加同名 `.txt`(Danbooru 风格小写标签,逗号分隔)。

In [ ]:
RUN_NAME = 'my_anima_lora'      #@param {type:'string'}
TRAIN_ROOT = Path(MODELS_DIR).parent / 'train_data' / RUN_NAME
LOCAL_DATA = Path('/content/train_data')
if not TRAIN_ROOT.is_dir():
    raise FileNotFoundError('Put the dataset in: ' + str(TRAIN_ROOT))
shutil.rmtree(LOCAL_DATA, ignore_errors=True)
shutil.copytree(TRAIN_ROOT, LOCAL_DATA)
img_exts = {'.png', '.jpg', '.jpeg', '.webp'}
imgs = [p for p in LOCAL_DATA.iterdir() if p.suffix.lower() in img_exts]
no_caption = [p for p in imgs if not p.with_suffix('.txt').exists()]
print('images:', len(imgs), '| missing captions:', len(no_caption))
assert imgs, 'No images found'
TRIGGER_FOR_MISSING = ''        #@param {type:'string'}
if no_caption and TRIGGER_FOR_MISSING:
    for p in no_caption:
        p.with_suffix('.txt').write_text(TRIGGER_FOR_MISSING)
    print('filled captions with trigger word:', TRIGGER_FOR_MISSING)
elif no_caption:
    raise RuntimeError('Some images have no caption. Add .txt files or set TRIGGER_FOR_MISSING.')

## (v9) 开始训练(后台运行)
- 训练开始前会让 ComfyUI worker 卸载模型释放显存。**训练期间不要从本机提交远程采样任务**,T4 的 16GB 放不下两件事;
- 训练在后台进程里运行,这一格会立刻返回;日志和 `status.json` 每 30 秒同步到 `MyDrive/ComfyUI/remote_train/<RUN_NAME>/`;
- 训练期间空闲计时器视为「有任务」,不会释放 runtime;训练结束后重新开始计时(`IDLE_MINUTES`);
- 每个 epoch 的 LoRA 会复制到 `models/loras/`,远程采样可以直接选用;
- 默认不转换格式:文档说明仅 DiT 的 LoRA 一般无需转换。如果 ComfyUI 加载后效果不对,再用 sd-scripts 里的 convert 脚本处理。

In [ ]:
import threading
RESOLUTION = 512
RANK = 16
LR = '5e-5'
EPOCHS = 10
REPEATS = 10
BLOCKS_TO_SWAP = 0          # OOM 时调大(最大 26),会变慢
PRECISION = 'bf16'          # T4 没有原生 bf16;报错或 loss=NaN 时改 'fp16'

if globals().get('TRAIN_PROC') is not None and TRAIN_PROC.poll() is None:
    raise RuntimeError('Training is already running.')

# free the GPU held by the ComfyUI worker
try:
    requests.post('http://127.0.0.1:8188/free', json={'unload_models': True, 'free_memory': True}, headers=worker_headers, timeout=10)
    time.sleep(3)
except requests.RequestException as error:
    print('Could not ask ComfyUI to free memory:', type(error).__name__)
import torch
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f'free VRAM: {free_gb:.1f} GB')
if free_gb < 10:
    print('Warning: less than 10 GB free. Consider BLOCKS_TO_SWAP > 0 or restarting the worker.')

TRAIN_OUT = Path('/content/train_out'); TRAIN_OUT.mkdir(exist_ok=True)
DRIVE_RUN_DIR = Path(MODELS_DIR).parent / 'remote_train' / RUN_NAME
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
LORA_DIR = model_root / 'loras'; LORA_DIR.mkdir(parents=True, exist_ok=True)

Path('/content/train_dataset.toml').write_text(f'''
[general]
shuffle_caption = false
caption_extension = ".txt"

[[datasets]]
resolution = {RESOLUTION}
batch_size = 1
enable_bucket = true
min_bucket_reso = 384
max_bucket_reso = 768
bucket_reso_steps = 64

  [[datasets.subsets]]
  image_dir = "{LOCAL_DATA}"
  num_repeats = {REPEATS}
''')

cmd = [
    str(VENV / 'bin' / 'accelerate'), 'launch', '--num_cpu_threads_per_process', '1', '--mixed_precision', PRECISION,
    'anima_train_network.py',
    f'--pretrained_model_name_or_path={DIT_PATH}', f'--qwen3={QWEN_PATH}', f'--vae={VAE_PATH}',
    '--dataset_config=/content/train_dataset.toml',
    f'--output_dir={TRAIN_OUT}', f'--output_name={RUN_NAME}', '--save_model_as=safetensors',
    '--network_module=networks.lora_anima', f'--network_dim={RANK}', f'--network_alpha={RANK}', '--network_train_unet_only',
    f'--learning_rate={LR}', '--optimizer_type=AdamW8bit', '--lr_scheduler=constant',
    '--timestep_sampling=sigmoid', '--discrete_flow_shift=1.0',
    f'--max_train_epochs={EPOCHS}', '--save_every_n_epochs=1', f'--mixed_precision={PRECISION}',
    '--gradient_checkpointing', '--cache_latents', '--cache_text_encoder_outputs', '--qwen_image_vae_2d', '--seed=42',
]
if BLOCKS_TO_SWAP > 0:
    cmd += [f'--blocks_to_swap={BLOCKS_TO_SWAP}', '--block_swap_h2d_only']

TRAIN_LOG = Path('/content/train.log')
TRAIN_PROC = subprocess.Popen(cmd, cwd=SD_DIR, stdout=open(TRAIN_LOG, 'w'), stderr=subprocess.STDOUT)

def _last_line(path):
    try:
        text = path.read_text(errors='replace')
    except OSError:
        return ''
    parts = [p for p in re.split(r'[\r\n]+', text) if p.strip()]
    return parts[-1][-300:] if parts else ''

def _sync_train_state(proc, state):
    def publish_log():
        try:
            shutil.copyfile(TRAIN_LOG, DRIVE_RUN_DIR / 'train.log')
        except OSError as error:
            print('log sync failed:', type(error).__name__)
    def publish_loras():
        for f in TRAIN_OUT.glob('*.safetensors'):
            dest = LORA_DIR / f.name
            if not dest.exists() or dest.stat().st_mtime < f.stat().st_mtime:
                shutil.copyfile(f, dest)
    while True:
        running = proc.poll() is None
        try:
            publish_log(); publish_loras()
            (DRIVE_RUN_DIR / 'status.json').write_text(json.dumps({
                'run': state['run'], 'state': 'running' if running else ('done' if proc.returncode == 0 else 'failed'),
                'returncode': proc.returncode, 'last_line': _last_line(TRAIN_LOG), 'updated': time.strftime('%Y-%m-%d %H:%M:%S'),
                'loras': sorted(p.name for p in TRAIN_OUT.glob('*.safetensors')),
            }, ensure_ascii=False, indent=2))
        except OSError as error:
            print('status sync failed:', type(error).__name__)
        if not running:
            return
        time.sleep(30)

TRAIN_SYNC = threading.Thread(target=_sync_train_state, args=(TRAIN_PROC, {'run': RUN_NAME}), daemon=True)
TRAIN_SYNC.start()
print('Training started in background. Log:', TRAIN_LOG, '| Drive status:', DRIVE_RUN_DIR / 'status.json')

In [ ]:
# 查看训练进度(可重复运行)
print('running' if TRAIN_PROC.poll() is None else f'exited with {TRAIN_PROC.returncode}')
print(_last_line(TRAIN_LOG))
print(sorted(p.name for p in TRAIN_OUT.glob('*.safetensors')))

In [ ]:
# 需要中止训练时运行
TRAIN_PROC.terminate()

## 本机使用统一提交接口
下载本次生成的 config.json，放回本机插件目录，重启本机 ComfyUI 并刷新网页。
普通入口 Remote Submit Diffusion (Latent) 与 SeedVR2 简化入口共用同一提交协议和 Remote Fetch Latent。
填写 model_name 和采样参数，连接对应 VAE 编码的 latent。需要文本条件的模型另外连接 positive/negative；SeedVR2 可留空，云端自动组装原生条件。
不需要选择模型类型或模型目录。模型名在 checkpoints 和 diffusion_models 中自动查找；同名文件有歧义时可在名字前加目录前缀。
普通示例：Remote-Unified-Submit-Fetch.json。SeedVR2 VAE 示例：Remote-Unified-SeedVR2.json；专用入口只简化输入及默认参数，后端处理共用。
持续队列：xiaobai-il-continuous-v8.json，点击控制器的 Start pipeline。首次设 total_images=2，max_in_flight=3。
本次云端协议已升级，必须使用 v8 后端。旧运行时请先处理完已有任务再关闭，重新启动 v8。

Anima 采样:模型名填 `anima-base-v1.0.safetensors`(在 diffusion_models 中查找)。LoRA 训练完成后会出现在 Drive 的 `models/loras/`。


## 手动立即释放或关闭定时器
需要立即释放时，新建代码格运行：
```python
from google.colab import runtime
runtime.unassign()
```
仅关闭自动定时器、保留运行时：
```python
idle_release_stop.set()
```
自动释放后远程服务不可用；重新启动 Notebook 后要重新下载并更新本机 `config.json`。


In [ ]:
!tail -n 60 -F /content/remote_diffusion_worker.log /content/remote_diffusion_tunnel.log

In [ ]:
idle_release_stop.set()